## Carga de documento CSV (Enero-2026) 
Se carga la primera quincena de enero de 2026 (`01-2026_Q1.csv`).

In [ ]:
import pandas as pd

ruta = '../datos/crudos/01-2026_Q1.csv'

df = pd.read_csv(ruta, encoding='utf-8-sig')

print(df.shape)
df.head()

(496726, 15)


,producto,presentacion,marca,categoria,catalogo,precio,fecha_registro,cadena_comercial,giro,nombre_comercial,direccion,estado,municipio,latitud,longitud
0,Aceite,Botella 850 Ml. Vegetal,Ave,Aceites y Grasas Veg. Comestibles,PACIC,28.0,2026/01/02,Chedraui,Supermercado / Tienda de Autoservicio,Chedraui Sucursal Aguascalientes,Blvd. Jose Ma. Chavez 1809. Entre Valente Quin...,Aguascalientes,Aguascalientes,21.854349,-102.294259
1,Aceite,Botella 875 Ml. Soya,Imperial,Aceites y Grasas Veg. Comestibles,PACIC,34.5,2026/01/02,Chedraui,Supermercado / Tienda de Autoservicio,Chedraui Sucursal Aguascalientes,Blvd. Jose Ma. Chavez 1809. Entre Valente Quin...,Aguascalientes,Aguascalientes,21.854349,-102.294259
2,Arroz,Bolsa 900 Gr. Super Extra. Grano Largo,Chedraui,Arroz y Cereales Preparados,PACIC,17.5,2026/01/02,Chedraui,Supermercado / Tienda de Autoservicio,Chedraui Sucursal Aguascalientes,Blvd. Jose Ma. Chavez 1809. Entre Valente Quin...,Aguascalientes,Aguascalientes,21.854349,-102.294259
3,Arroz,Bolsa 900 Gr. Super Extra. Verde,Schettino,Arroz y Cereales Preparados,PACIC,20.0,2026/01/02,Chedraui,Supermercado / Tienda de Autoservicio,Chedraui Sucursal Aguascalientes,Blvd. Jose Ma. Chavez 1809. Entre Valente Quin...,Aguascalientes,Aguascalientes,21.854349,-102.294259
4,Atún,Lata 140 Gr. Desmenuzado en Agua. Standard. Si...,Fresh Label,Pescados y Mariscos en Conserva,PACIC,9.0,2026/01/02,Chedraui,Supermercado / Tienda de Autoservicio,Chedraui Sucursal Aguascalientes,Blvd. Jose Ma. Chavez 1809. Entre Valente Quin...,Aguascalientes,Aguascalientes,21.854349,-102.294259


Revisamos el formato de las variables y revisar si hay datos faltantes.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 496726 entries, 0 to 496725
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   producto          496726 non-null  str    
 1   presentacion      496726 non-null  str    
 2   marca             496726 non-null  str    
 3   categoria         496726 non-null  str    
 4   catalogo          496726 non-null  str    
 5   precio            496726 non-null  float64
 6   fecha_registro    496726 non-null  str    
 7   cadena_comercial  496726 non-null  str    
 8   giro              496726 non-null  str    
 9   nombre_comercial  496726 non-null  str    
 10  direccion         496726 non-null  str    
 11  estado            496726 non-null  str    
 12  municipio         496726 non-null  str    
 13  latitud           496726 non-null  float64
 14  longitud          496726 non-null  float64
dtypes: float64(3), str(12)
memory usage: 56.8 MB


**Revisamos si hay duplicados**

Podemos observar que el metodo nos arroja "0"

In [4]:
df.duplicated().sum()

np.int64(0)

Revisamos que columnas no tienen "catalogo"

In [5]:
columnas_sin_catalogo = [c for c in df.columns if c != 'catalogo']
df.duplicated(subset=columnas_sin_catalogo).sum()

np.int64(9346)

Con .isna vamos a revisar si hay valores faltantes o nulos (como NaN, None o NaT en fechas)

In [6]:
df.isna().sum()

producto            0
presentacion        0
marca               0
categoria           0
catalogo            0
precio              0
fecha_registro      0
cadena_comercial    0
giro                0
nombre_comercial    0
direccion           0
estado              0
municipio           0
latitud             0
longitud            0
dtype: int64

Revisamos el rango de fechas 

In [8]:
df['fecha_registro'].agg(['min', 'max'])

min    2026/01/02
max    2026/01/15
Name: fecha_registro, dtype: str

**Revisar duplicados ocultos:**

In [9]:
(df['marca']=='S/M').mean()

np.float64(0.3588094845045357)

Podemos observar que el minimo es una latitud negativa, por lo que no se encuentra en México, deduzco que es un error de captura ya que los demas datos relacionados a este lugar estan correctos de que pertenecen en México

In [10]:
df[['latitud','longitud']].describe()

,latitud,longitud
count,496726.000000,496726.000000
mean,21.037217,-99.601504
std,3.401212,5.142109
min,-19.440087,-117.041818
25%,19.345444,-101.633559
50%,19.558491,-99.194678
75%,21.163330,-98.236332
max,32.535127,99.112880


Cree un filtro para observar que localidades aparecen con latitud negativa:

In [11]:
fuera_de_mexico = df.loc[
    (df['latitud'] < 14) | (df['latitud'] > 33) | (df['longitud'] > -86) | (df['longitud'] < -119),
    ['estado', 'municipio', 'cadena_comercial', 'latitud', 'longitud']
]
len(fuera_de_mexico)
fuera_de_mexico

,estado,municipio,cadena_comercial,latitud,longitud
126332,Ciudad de México,Gustavo a. Madero,Tortillerias Tradicionales,19.461110,99.112880
171028,Ciudad de México,Venustiano Carranza,Tortillerias Tradicionales,-19.440087,-99.107246
175195,Ciudad de México,Venustiano Carranza,Tortillerias Tradicionales,-19.440087,-99.107246


Modifique que todas las latitudes sean positivas y todas las longitudes sean negativas.

In [12]:
df['latitud'] = df['latitud'].abs()
df['longitud'] = -df['longitud'].abs()

In [13]:
df[['latitud', 'longitud']].describe()

,latitud,longitud
count,496726.000000,496726.000000
mean,21.037374,-99.601903
std,3.400243,5.134373
min,16.736944,-117.041818
25%,19.345444,-101.633559
50%,19.558491,-99.194678
75%,21.163330,-98.236332
max,32.535127,-86.824104


In [14]:
fuera = (df['latitud'] < 14) | (df['latitud'] > 33) | (df['longitud'] > -86) | (df['longitud'] < -119)
fuera.sum()

np.int64(0)

**Función de limpieza**

In [17]:
import numpy as np
from pathlib import Path

def limpiar_archivo(ruta):
    """Lee un archivo de Profeco y lo limpia. Devuelve (df_limpio, resumen)."""
    df = pd.read_csv(ruta, encoding='utf-8-sig')
    n_original = len(df)

    # Errores de signo en coordenadas (se cuentan antes de corregirlos)
    errores_signo = ((df['latitud'] < 0) | (df['longitud'] > 0)).sum()

    # 1) Duplicados escondidos: se ignora 'catalogo' y se conserva la fila de Básicos
    columnas_sin_catalogo = [c for c in df.columns if c != 'catalogo']
    df['_es_basico'] = df['catalogo'] == 'Básicos'
    df = (df.sort_values('_es_basico', ascending=False)
            .drop_duplicates(subset=columnas_sin_catalogo, keep='first')
            .drop(columns='_es_basico')
            .sort_index())
    n_duplicados = n_original - len(df)

    # 2) Marca: 'S/M' pasa a 'Sin marca'
    df['marca'] = df['marca'].replace('S/M','Sin marca')

    # 3) Fecha: de texto a datetime
    df['fecha_registro'] = pd.to_datetime(df['fecha_registro'])

    # 4) Coordenadas: en México la latitud es positiva y la longitud negativa
    df['latitud'] = df['latitud'].abs()
    df['longitud'] = -df['longitud'].abs()

    # Red de seguridad: si algo sigue fuera de México, se deja vacío
    fuera = (df['latitud'] < 14) | (df['latitud'] > 33) | (df['longitud'] > -86) | (df['longitud'] < -119)
    df.loc[fuera, ['latitud', 'longitud']] = np.nan

    df = df.reset_index(drop=True)

    resumen = {
        'archivo': Path(ruta).name,
        'registros_originales': n_original,
        'duplicados_eliminados': n_duplicados,
        'registros_finales': len(df),
        'coordenadas_corregidas': int(errores_signo),
        'coordenadas_vacias': int(df['latitud'].isna().sum()),
        'pct_sin_marca': float(round((df['marca'] == 'Sin marca').mean() * 100, 1)),
    }
    return df, resumen

In [16]:
df_enero, resumen = limpiar_archivo('../datos/crudos/01-2026_Q1.csv')
resumen

{'archivo': '01-2026_Q1.csv',
 'registros_originales': 496726,
 'duplicados_eliminados': 9346,
 'registros_finales': 487380,
 'coordenadas_corregidas': 3,
 'coordenadas_vacias': 0,
 'pct_sin_marca': np.float64(35.6)}

In [20]:
df_enero.to_parquet('../datos/limpios/01-2026_Q1.parquet', index=False)

In [19]:
prueba = pd.read_parquet('../datos/limpios/01-2026_Q1.parquet')
print(prueba.shape)
prueba['fecha_registro'].dtype

(487380, 15)


dtype('<M8[us]')

**Limpiar los 20 archivos con try/except**

Ahora aplicamos la función a todos los archivos de datos/crudos, uno por uno. Con 20 archivos, es posible que alguno traiga un problema distinto (otro nombre de columna, un formato de fecha raro). Con try/except, el programa anota el error y sigue con el siguiente.

In [ ]:
from pathlib import Path

archivos = sorted(Path('../datos/crudos').glob('*.csv'))
print(len(archivos))      # debe salir 20

#Listas para acumular métricas de ejecución e incidencias por archivo
resumenes = []
errores = []

# Procesar cada archivo CSV de forma individual
for ruta in archivos:
    #es la ruta de salida
    destino = Path('../datos/limpios') / (ruta.stem + '.parquet')
    try:
        #retorna el DataFrame limpio y un diccionario resumen
        df_limpio, resumen = limpiar_archivo(ruta)

        #Guarda el DataFrame en formato Parquet sin incluir el indice por defecto de pandas
        df_limpio.to_parquet(destino, index=False)              

        #Registra el resumen del archivo procesado y muestra el total de filas resultantes
        resumenes.append(resumen)
        print('OK   ', ruta.name, f"{resumen['registros_finales']:,}")
    except Exception as e:
        #Captura cualquier fallo para evitar que se detenga todo el lote y guarda el detalle
        errores.append((ruta.name, str(e)))
        print('ERROR', ruta.name, e)

20
OK    01-2026_Q1.csv 487,380
OK    01-2026_Q2.csv 618,933
OK    02-2026_Q1.csv 487,671
OK    02-2026_Q2.csv 539,249
OK    03-2026_Q1.csv 549,564
OK    03-2026_Q2.csv 597,073
OK    04-2026_Q1.csv 472,634
OK    04-2026_Q2.csv 588,087
OK    05-2026_Q1.csv 472,315
OK    05-2026_Q2.csv 525,212
OK    06-2026_Q1.csv 612,756
OK    06-2026_Q2.csv 624,124
OK    07-2026_Q1.csv 656,063
OK    07-2026_Q2.csv 734,137
OK    08-2026_Q1.csv 626,892
OK    08-2026_Q2.csv 648,148
OK    09-2026_Q1.csv 594,438
OK    09-2026_Q2.csv 526,188
OK    10-2026_Q1.csv 607,760
OK    10-2026_Q2.csv 542,514


comprobamos que sea la cantidad de archivos limpios esperada:

In [22]:
print(len(resumenes), 'archivos limpiados')
print(errores)

20 archivos limpiados
[]


Convertir los resumenes en DataFrame, obtenemos las modificaciones que se hicieron

In [23]:
resumen_limpieza = pd.DataFrame(resumenes)
resumen_limpieza

,archivo,registros_originales,duplicados_eliminados,registros_finales,coordenadas_corregidas,coordenadas_vacias,pct_sin_marca
0,01-2026_Q1.csv,496726,9346,487380,3,0,35.6
1,01-2026_Q2.csv,630371,11438,618933,2,0,34.1
2,02-2026_Q1.csv,496645,8974,487671,4,0,34.5
3,02-2026_Q2.csv,547811,8562,539249,3,0,33.7
4,03-2026_Q1.csv,558146,8582,549564,2,0,34.4
5,03-2026_Q2.csv,606522,9449,597073,3,0,35.2
6,04-2026_Q1.csv,480842,8208,472634,2,0,33.9
7,04-2026_Q2.csv,597634,9547,588087,4,163,34.1
8,05-2026_Q1.csv,480010,7695,472315,2,323,33.3
9,05-2026_Q2.csv,533214,8002,525212,2,305,32.6


Calcular los totales de las tres columnas de registros, con **.sum()**

In [24]:
resumen_limpieza[['registros_originales', 'duplicados_eliminados', 'registros_finales']].sum()

registros_originales     11695588
duplicados_eliminados      184450
registros_finales        11511138
dtype: int64

In [25]:
resumen_limpieza.to_csv('../datos/procesados/resumen_limpieza.csv', index=False)

Resumen de limpieza:

In [26]:
resumen_limpieza.sort_values('registros_finales', ascending=False)[['archivo', 'registros_finales']].head(5)

,archivo,registros_finales
13,07-2026_Q2.csv,734137
12,07-2026_Q1.csv,656063
15,08-2026_Q2.csv,648148
14,08-2026_Q1.csv,626892
11,06-2026_Q2.csv,624124


In [27]:
resumen_limpieza['pct_sin_marca'].agg(['min', 'max'])

min    30.9
max    35.6
Name: pct_sin_marca, dtype: float64

In [28]:
resumen_limpieza[['archivo', 'coordenadas_corregidas', 'coordenadas_vacias']]

,archivo,coordenadas_corregidas,coordenadas_vacias
0,01-2026_Q1.csv,3,0
1,01-2026_Q2.csv,2,0
2,02-2026_Q1.csv,4,0
3,02-2026_Q2.csv,3,0
4,03-2026_Q1.csv,2,0
5,03-2026_Q2.csv,3,0
6,04-2026_Q1.csv,2,0
7,04-2026_Q2.csv,4,163
8,05-2026_Q1.csv,2,323
9,05-2026_Q2.csv,2,305
